In [3]:
import numpy as np
import pandas as pd

# ----------------------------
# 1. Load current files
# ----------------------------
print("Loading current files...")
pheno = pd.read_csv('ukbb_pheno.csv')
data = np.load('ukbb_features.npz')

X = data['features']  # (n, p)
conn_eids = data['subject_ids'].astype(str)
feature_names = data['feature_names']
region_names = data['region_names']

pheno['eid'] = pheno['eid'].astype(str)

# ----------------------------
# 2. Find common subjects and sort by EID (deterministic order)
# ----------------------------
common_eids = sorted(set(pheno['eid']) & set(conn_eids))
print(f"Common subjects: {len(common_eids)}")

if len(common_eids) != len(pheno) or len(common_eids) != X.shape[0]:
    raise ValueError("Unexpected subject mismatch — should not happen if files are from same alignment run.")

# ----------------------------
# 3. Reorder BOTH datasets by sorted EID
# ----------------------------
# Build index maps
pheno_eid_to_idx = {eid: i for i, eid in enumerate(pheno['eid'])}
conn_eid_to_idx = {eid: i for i, eid in enumerate(conn_eids)}

# Get indices for sorted EID order
pheno_indices = [pheno_eid_to_idx[eid] for eid in common_eids]
conn_indices = [conn_eid_to_idx[eid] for eid in common_eids]

# Reorder
pheno_final = pheno.iloc[pheno_indices].reset_index(drop=True)
X_final = X[conn_indices]
eids_final = common_eids  # already sorted

# ----------------------------
# 4. Final sanity check
# ----------------------------
assert list(pheno_final['eid']) == eids_final
assert X_final.shape[0] == len(eids_final)
print("✅ Perfect alignment confirmed.")

# ----------------------------
# 5. OVERWRITE your files with the sorted, aligned versions
# ----------------------------
print("Saving final, perfectly aligned files...")

# Save phenotypes
pheno_final.to_csv('ukbb_pheno.csv', index=False)

# Save connectivity
np.savez_compressed(
    'ukbb_features.npz',
    features=X_final,
    subject_ids=eids_final,
    feature_names=feature_names,
    region_names=region_names
)

# Save EID list
np.savetxt('ukbb_eids.txt', eids_final, fmt='%s')

print("✅✅✅ FINAL FILES SAVED — NOW PERFECTLY ALIGNED AND SORTED BY EID ✅✅✅")

Loading current files...
Common subjects: 16340
✅ Perfect alignment confirmed.
Saving final, perfectly aligned files...
✅✅✅ FINAL FILES SAVED — NOW PERFECTLY ALIGNED AND SORTED BY EID ✅✅✅
